Выгрузим данные и лучшую нашу модель

In [2]:
import joblib
import pandas as pd
from sklearn.metrics import mean_absolute_error, r2_score

X_test = pd.read_parquet('../data/future_testing/X_test_final.parquet')
y_test = pd.read_parquet('../data/future_testing/y_test_final.parquet')

model = joblib.load('../models/lightgbm_v1.pkl')
display(X_test.head())
print(f"Размер таблицы: {X_test.shape}")

display(y_test.head())
print(f"Размер таблицы: {y_test.shape}")

,place,platform,agent,like,dislike,share,bookmark,click_on_author,open_comments,duration,...,author_video_count,user_watch_count,engagement_score,author_avg_target,author_avg_engagement,author_popularity,user_avg_target,user_avg_engagement,user_activity,video_duration_type
10117391,0,0,0,0,0,0,0,0,0,52,...,3059,930,0,0.513505,0.091533,3059,0.651635,0.000000,930,3
26635015,2,0,0,0,0,0,0,0,0,45,...,3097,741,0,0.542583,0.022603,3097,0.291384,0.005398,741,3
37339584,0,0,0,0,0,0,0,0,0,73,...,25340,57,0,0.562741,0.004223,25340,0.294076,0.000000,57,3
22954482,0,0,0,0,0,0,0,0,0,60,...,26299,3388,0,0.462244,0.022472,26299,0.727800,0.000885,3388,3
30333544,2,0,0,0,0,0,0,0,0,16,...,65473,112,0,0.601148,0.026133,65473,0.773216,0.000000,112,2


Размер таблицы: (1000000, 23)


,target
10117391,1.000000
26635015,0.111111
37339584,0.232877
22954482,0.016667
30333544,0.437500


Размер таблицы: (1000000, 1)


In [3]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Средняя ошибка (MAE): {mae:.4f}")
print(f"Коэффициент детерминации (R2): {r2:.4f}")

Средняя ошибка (MAE): 0.2993
Коэффициент детерминации (R2): 0.2694


In [5]:
y_pred_ratio = model.predict(X_test)

comparison_df = pd.DataFrame({
    'Real_Ratio': y_test.values.flatten(),
    'Predicted_Ratio': y_pred_ratio,
    'Duration_Sec': X_test['duration'].values
})

comparison_df['Real_Seconds'] = comparison_df['Real_Ratio'] * comparison_df['Duration_Sec']
comparison_df['Predicted_Seconds'] = comparison_df['Predicted_Ratio'] * comparison_df['Duration_Sec']
comparison_df['Error_Seconds'] = (comparison_df['Real_Seconds'] - comparison_df['Predicted_Seconds']).abs()

print("Сравнение реального времени просмотра и предсказанного (в секундах):")
display(comparison_df[['Real_Seconds', 'Predicted_Seconds', 'Error_Seconds', 'Duration_Sec']].head(100))

Сравнение реального времени просмотра и предсказанного (в секундах):


,Real_Seconds,Predicted_Seconds,Error_Seconds,Duration_Sec
0,52.0,34.626630,17.373370,52
1,5.0,9.380267,4.380267,45
2,17.0,22.267674,5.267674,73
3,1.0,39.729454,38.729454,60
4,7.0,10.200547,3.200547,16
...,...,...,...,...
95,2.0,20.335862,18.335862,81
96,50.0,44.663555,5.336445,51
97,5.0,27.307219,22.307219,36
98,12.0,9.508084,2.491916,12


In [7]:
mae_seconds = comparison_df['Error_Seconds'].mean()
print(f"Средняя ошибка: {mae_seconds:.2f} секунд")

median_error_sec = comparison_df['Error_Seconds'].median()
print(f"Медианная ошибка: {median_error_sec:.2f} секунд")

Средняя ошибка: 15.03 секунд
Медианная ошибка: 10.88 секунд
